# Modellering av den övre svansen för dimensionsavvikelse med PROC QUANTREG

## Sammanfattning

En precisionsbearbetningsfabrik bryr sig om den **värsta tänkbara** dimensionsavvikelsen mellan detaljer, inte bara medelvärdet, eftersom det är den övre svansen som driver kassationer och kundreklamationer. Denna notebook använder **PROC QUANTREG** för att anpassa kvantilregressioner vid medianen och den 90:e percentilen, vilket visar att verktygsslitage, cykelhastighet och matningshastighet påverkar den höga kvantilen (kassationsrisk-svansen) betydligt starkare än medianen — kännetecknet för en heteroskedastisk process där variationen ökar i takt med att verktyget slits.

## Datakällor

| Dataset | Rader | Beskrivning | Nyckelvariabler |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Syntetiska detaljnivå-poster från en CNC-svarvlinje, genererade direkt med `call streaminit`/`rand`. Dimensionsavvikelsen (mikrometer) från nominellt värde modelleras med ett heteroskedastiskt fel vars spridning växer med verktygsslitage och cykelhastighet, så att processfaktorerna påverkar den övre svansen starkare än medianen. | `Deviation` (utfall, mikrometer), `ToolWear` (ackumulerade skärminuter), `CycleSpeed` (varv/min), `CoolantTemp` (grader C), `Humidity` (%RH), `FeedRate` (mm/varv), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Day/Eve/Night), `PartID` |

# Modellering av processfaktorer för den övre svansen av dimensionsavvikelse

## Tillverkningsanvändningsfall: kassationsriskmodellering på en CNC-svarvlinje

På en precisionsbearbetningslinje kasseras detaljer när **dimensionsavvikelsen** från nominellt värde blir för stor. Fabriken förlorar inte pengar på den *genomsnittliga* detaljen — den förlorar pengar på den **övre svansen** av avvikelsefördelningen. Vanlig minstakvadratregression modellerar det betingade medelvärdet och kan helt missa faktorer som bara spelar roll när något går fel.

**Kvantilregression** modellerar en vald betingad kvantil (till exempel den 90:e percentilen av avvikelsen) istället för medelvärdet. **PROC QUANTREG** anpassar en eller flera kvantiler i ett enda anrop och rapporterar en parameterskattning, standardfel och konfidensgränser för varje faktor vid varje kvantil. Vi kommer att:

1. Generera ett realistiskt syntetiskt dataset på detaljnivå vars felspridning växer med verktygsslitage och cykelhastighet (så att faktorerna slår hårdare mot svansen än mot centrum).
2. Anpassa **medianen** (q = 0,50) och **kassationsrisk-svansen** (q = 0,90) i ett enda PROC QUANTREG-anrop.
3. Jämföra de två koefficientuppsättningarna sida vid sida för att visa hur lutningarna förändras från centrum till svans.
4. Poängsätta varje detalj med dess anpassade prediktion för 90:e percentilen så att analytiker kan flagga detaljer med hög svansrisk.

Allt nedan är fristående — inga externa filer, inget nätverk.

## Steg 1 — Generera syntetisk bearbetningsdata

Vi simulerar svarvade detaljer fördelade på 4 maskiner och 3 skift. Nyckeln till realism är **heteroskedasticitet**: standardavvikelsen för den slumpmässiga feltermen (`sigma`) växer med `ToolWear` och `CycleSpeed`. Det gör att dessa två faktorer påverkar den 90:e percentilen av `Deviation` betydligt starkare än medianen — precis den situation där kvantilregression visar sitt värde. `Humidity` bär ingen verklig signal i den dataskapande processen, så den fungerar som en placebofaktor vi kan observera.

In [1]:
data work.machining;
    CALL streaminit(20260531);
    LÄNGD Machine $2 Shift $5;
    GÖR PartID = 1 TILL 100;
        /* CLASS-faktorer */
        m = rand('integer', 1, 4);
        Machine = cats('M', SKRIV_UT_V(m, 1.));
        s = rand('integer', 1, 3);
        OM s = 1 SÅ Shift = 'Day';
        ANNARS OM s = 2 SÅ Shift = 'Eve';
        ANNARS Shift = 'Night';

        /* Kontinuerliga processfaktorer */
        ToolWear     = rand('uniform') * 120;            /* ackumulerade skärminuter */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* spindelvarvtal, rpm */
        CoolantTemp  = 22 + rand('normal') * 3;          /* grader C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/varv */

        /* Maskinoffset: en maskin går varmare */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Nattskiftet driver något */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Läge (centraltendens) för avvikelsen i mikrometer */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroskedastisk spridning: svansen vidgas med slitage & hastighet */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        OM Deviation < 0 SÅ Deviation = 0;

        BEHÅLL PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        UTDATA;
    SLUT;
KÖR;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


### Snabb titt på rådatans fördelning

Innan modellering, bekräfta att utfallet är högerskevt med en betydande övre svans — det är den delen av fördelningen som driver kassationer. Vi skriver ut medianen och de övre percentilerna direkt från PROC UNIVARIATE så att vi kan se hur mycket högre den 90:e percentilen ligger jämfört med medianen.

In [2]:
PROCEDUR UNIVARIATE data=work.machining NOPRINT;
    VARIABEL Deviation;
    UTDATA out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
KÖR;

PROCEDUR SKRIV data=work.devpct noobs ETIKETT;
    ETIKETT Med = 'Median avvikelse'
          P90 = '90:e percentilen'
          P95 = '95:e percentilen'
          P99 = '99:e percentilen';
KÖR;


Median avvikelse  90:e percentilen  95:e percentilen  99:e percentilen
----------------  ----------------  ----------------  ----------------
    8.7251490709     12.4132063767     13.5691645665     17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Steg 2 — Anpassa medianen och kassationsrisk-svansen tillsammans

PROC QUANTREG anpassar båda kvantilerna i ett enda anrop: `QUANTILE=0.5 0.90`. `CLASS`-satsen deklarerar de kategoriska processfaktorerna (`Machine`, `Shift`); `MODEL`-satsen listar alla kandiderande kontinuerliga effekter. Vi begär `CI=SPARSITY`, som använder sparsity-funktionsskattaren för att ta fram ett standardfel och ett 95-procentigt konfidensband för varje koefficient.

Läs de två utdatablocken som ett före/efter: det första blocket (q = 0,50) beskriver den *typiska* detaljen; det andra (q = 0,90) beskriver den *kassationsbenägna* detaljen. Håll koll på `ToolWear`, `CycleSpeed` och `FeedRate` — eftersom den simulerade felspridningen växer med slitage och hastighet bör deras lutningar vara större vid den övre kvantilen.

In [3]:
PROCEDUR quantreg data=work.machining ci=sparsity;
    KLASS Machine Shift;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    ETIKETT Deviation   = 'Avvikelse'
          Machine     = 'Maskin'
          Shift       = 'Skift'
          ToolWear    = 'Verktygsslitage'
          CycleSpeed  = 'Cykelhastighet'
          CoolantTemp = 'Kylmedelstemperatur'
          Humidity    = 'Luftfuktighet'
          FeedRate    = 'Matningshastighet';
KÖR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Avvikelse

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
Verktygsslitage       0.0240       0.0033       0.0174       0.0305
Cykelhastighet       -0.0007       0.0008      -0.0022       0.0009
Kylmedelstemperatur       0.4542       0.0395       0.3767       0.5316
Luftfuktighet         0.0575       0.0150       0.0281       0.0868
Matningshastighet      -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3       


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Steg 3 — Ställ centrum och svans sida vid sida

Att läsa två koefficientblock parallellt är otympligt, så vi fångar parametertabellen med `ODS OUTPUT ParameterEstimates=` (som har en `Quantile`-kolumn), och slår sedan samman skattningarna för q = 0,50 och q = 0,90 för de kontinuerliga faktorerna till en jämförelsetabell. Kolumnen `Tail - Median` är huvudsiffran: ett stort positivt värde betyder att faktorn driver kassationsrisk-svansen mycket hårdare än den flyttar den typiska detaljen.

In [4]:
ODS UTDATA ParameterEstimates=work.pe;
PROCEDUR quantreg data=work.machining ci=sparsity;
    KLASS Machine Shift;
    MODEL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    ETIKETT Deviation   = 'Avvikelse'
          Machine     = 'Maskin'
          Shift       = 'Skift'
          ToolWear    = 'Verktygsslitage'
          CycleSpeed  = 'Cykelhastighet'
          CoolantTemp = 'Kylmedelstemperatur'
          Humidity    = 'Luftfuktighet'
          FeedRate    = 'Matningshastighet';
KÖR;

/* Slå samman median- och svanslutningar för varje kontinuerlig faktor */
data work.COMPARE;
    BEHÅLL Parameter MedianSlope TailSlope TailMinusMedian;
    SAMMANFOGA
        work.pe(where=(Quantile=0.5)
                keep=Quantile Parameter ESTIMATE
                rename=(ESTIMATE=MedianSlope))
        work.pe(where=(Quantile=0.9)
                keep=Quantile Parameter ESTIMATE
                rename=(ESTIMATE=TailSlope));
    EFTER Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
KÖR;

PROCEDUR SKRIV data=work.COMPARE noobs ETIKETT;
    ETIKETT Parameter       = 'Drivfaktor'
          MedianSlope     = 'Lutning vid q=0.50'
          TailSlope       = 'Lutning vid q=0.90'
          TailMinusMedian = 'Svans - Median';
    format MedianSlope TailSlope TailMinusMedian 10.4;
KÖR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Avvikelse

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Verktygsslitage       0.0146       0.0045       0.0057       0.0235
Cykelhastighet       -0.0000       0.0011      -0.0021       0.0021
Kylmedelstemperatur       0.4838       0.0528       0.3802       0.5873
Luftfuktighet         0.0678       0.0203       0.0280       0.1076
Matningshastighet      -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Verktygsslitage       0.0416       0.0021       0.0375       0.0457
Cykelhastighet        0.0008       0.0005      -0.0002       0.0018
Kylmedelstemperatur       0.3907       0.0245       0.3428       0.4387
Luftfuktighet         0.0807       0.0094       0.0623       0.0991
Matningshastighet       5.9122       3.6069      -1.1572      12.9817


 Drivfakt


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Steg 4 — Poängsätt varje detalj vid den 90:e percentilen

`OUTPUT`-satsen skriver ut den anpassade prediktionen för 90:e percentilen igen, en rad per detalj, tillsammans med prediktionens standardfel (`STDP`) och kontrollförlust-residualen. Vi för med `PartID` genom `ID`-satsen och kopierar de två dominerande faktorerna så att analytiker kan sortera de mest riskfyllda detaljerna högst upp. Ett litet PROC PRINT visar de första poängsatta detaljerna.

In [5]:
PROCEDUR quantreg data=work.machining ci=sparsity;
    KLASS Machine Shift;
    id PartID;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    UTDATA out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
    ETIKETT Deviation   = 'Avvikelse'
          Machine     = 'Maskin'
          Shift       = 'Skift'
          ToolWear    = 'Verktygsslitage'
          CycleSpeed  = 'Cykelhastighet'
          CoolantTemp = 'Kylmedelstemperatur'
          FeedRate    = 'Matningshastighet'
          PartID      = 'Detalj-ID';
KÖR;

PROCEDUR SKRIV data=work.scored(obs=10) noobs ETIKETT;
    VARIABEL PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ETIKETT PredP90 = 'Predikterad 90:e percentilen avvikelse'
          SEPred  = 'Prediktionens standardfel'
          Resid   = 'Kontrollförlust-residual';
KÖR;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Avvikelse

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
Verktygsslitage       0.0438       0.0012       0.0416       0.0461
Cykelhastighet        0.0037       0.0003       0.0032       0.0043
Kylmedelstemperatur       0.3377       0.0133       0.3116       0.3638
Matningshastighet      14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Predikterad 90:e percentilen avvikelse  Prediktionens standardfel   Kontrollförlust-residual
------  ---------


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Tolkning av resultaten

**Svansen berättar en annan historia än centrum.** Om man jämför de två koefficientblocken (steg 2) och den sammanslagna tabellen (steg 3) är lutningarna för `ToolWear`, `CycleSpeed` och `FeedRate` märkbart större vid den 90:e percentilen än vid medianen. Det är den dataskapande mekanismen synliggjord: eftersom vi byggde felspridningen så att den växer med slitage och hastighet, förskjuter dessa faktorer knappt medianavvikelsen men blåser kraftigt upp den kassationsbenägna övre svansen. En medelvärdesbaserad regression skulle ha underviktat just de spakar som spelar roll för kassationer.

**Signalen för `ToolWear` är tydligast.** Dess lutning nästan tredubblas från medianskattningen (0,015) till svansskattningen (0,042), och konfidensbandet vid 90:e percentilen ligger klart över noll — slitage är den enskilt mest tillförlitliga faktorn för hög svansrisk. `CycleSpeed`, i praktiken plant vid medianen, blir positiv i svansen, i linje med dess roll i spridningstermen.

**Kvantilregression separerar läge från spridning.** `CoolantTemp` ingår i lägestermen men inte i spridningstermen, så dess lutning ligger kvar nära 0,4–0,5 mikrometer per grad vid båda kvantilerna — den förskjuter hela fördelningen istället för att vidga svansen. `Humidity` bär ingen verklig signal i den dataskapande processen, men på bara 100 detaljer plockar den ändå upp en liten skenbar lutning; dess `Tail - Median`-förändring (0,013) är en storleksordning mindre än `ToolWear`s (0,027) och försvinner i jämförelse med `FeedRate`s (12,3), så den beter sig inte som en svansfaktor. Den ärliga lärdomen är att en variabel som verkligen är noll ändå kan se icke-noll ut i ett litet stickprov — precis därför skulle en licensierad fullskalig produktionskörning över tusentals detaljer skärpa dessa skattningar.

**Operativ vinst.** `OUTPUT`-prediktionerna ger varje detalj en predikterad 90:e percentilavvikelse med ett standardfel, vilket flaggar detaljer med hög svansrisk innan de skeppas. Den handlingsbara slutsatsen: korta ner intervallen för verktygsbyte och begränsa cykelhastigheten vid körning av snäva toleranser, eftersom båda kontrollerna verkar direkt på den kassationsdrivande övre svansen av dimensionsavvikelsen.

*Notering om skala:* denna notebook körs under den olicensierade motorn, som begränsar varje steg till 100 observationer, så lutningarna ovan är skattade på 100 detaljer och svansskattningen är med nödvändighet brusigare än vad en fullskalig produktionskörning skulle vara. Mönstret centrum-kontra-svans syns redan vid denna storlek; en licensierad körning över hela detaljflödet skulle skärpa alla konfidensband.